# Final Exam — Practice: Full Walkthrough Solution

This notebook solves the practice exam **step by step**, explaining every decision and every line of code.

**Before running:** update the paths to `train.csv` and `test.csv` in the cell below.

---

## What this exam tests

| Problem | Concept | Key skill |
|---------|---------|----------|
| 1a | Load data | `np.genfromtxt`, shape checks |
| 1b | PCA | Normalise → covariance → `np.linalg.eig` → project |
| 1c | Visualise → choose degree | `plt.scatter`, judgment |
| 1d | Polynomial regression | Design matrix → normal equation → RMSE |
| 1e | Predict test data | Apply same pipeline consistently |
| 2d | Sample custom distribution | `np.random.default_rng`, inversion sampling |
| 2e | Verify variance numerically | `np.var`, loop over lambda values |

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Update these paths
TRAIN_PATH = r'C:\Pranay\Study Material\Masters\Methods and tools for AI applications\final example\train.csv'
TEST_PATH  = r'C:\Pranay\Study Material\Masters\Methods and tools for AI applications\final example\test.csv'

---
## Problem 1(a) — Load data, verify shape

train.csv has 6 columns (x0–x4 and t). test.csv has 5 columns (x0–x4 only).

In [2]:
# --- 1(a) ---
# np.genfromtxt reads CSV into a numpy array.
# skip_header=1 skips the column name row.

train_data = np.genfromtxt(TRAIN_PATH, delimiter=',', skip_header=1)
test_data  = np.genfromtxt(TEST_PATH,  delimiter=',', skip_header=1)

# Split features and targets
X_train = train_data[:, :5]   # columns 0-4, shape (m, 5)
t_train = train_data[:, 5]    # column 5,   shape (m,)
X_test  = test_data            # all 5 cols, shape (m_test, 5)

print('Training: X_train =', X_train.shape, '  t_train =', t_train.shape)
print('Test:     X_test  =', X_test.shape)
print()
print('First 3 training rows (X):')
print(X_train[:3].round(3))
print('First 3 targets (t):')
print(t_train[:3].round(3))

Training: X_train = (50, 5)   t_train = (50,)
Test:     X_test  = (10, 5)

First 3 training rows (X):
[[ 21.989  67.437  47.131 180.04  112.607]
 [ 18.424  51.316  38.672 141.127  86.56 ]
 [  7.093  19.719  13.96   53.116  32.579]]
First 3 targets (t):
[-69.68  -11.893  24.949]


---
## Problem 1(b) — Reduce dimension from 5 to 1

**Algorithm used: PCA (Principal Component Analysis)**

The 5 features are highly correlated — they all scale together (x1 ~ 3*x0, x3 ~ 8*x0, etc.).
PCA finds the single direction that captures the most variance.

**Why normalise first?**
x0 ranges ~0-30, x3 ranges ~0-200. Without normalisation x3 would dominate
just because of its scale, not because it carries more information.
Z-score normalisation puts all features on equal footing (mean=0, std=1).

In [ ]:
# --- 1(b) ---

# STEP 1: Normalise using TRAINING statistics only
mu_X    = X_train.mean(axis=0)   # mean of each feature column, shape (5,)
sigma_X = X_train.std(axis=0)    # std  of each feature column, shape (5,)

X_train_norm = (X_train - mu_X) / sigma_X   # shape (m, 5)

print('After normalisation:')
print('  means:', X_train_norm.mean(axis=0).round(10), ' (should be ~0)')
print('  stds: ', X_train_norm.std(axis=0).round(10),  ' (should be ~1)')

In [ ]:
# STEP 2: Covariance matrix — shape (5, 5)
# Captures how features vary together.
# Off-diagonal values near +/-1 mean features are highly correlated.
m = X_train_norm.shape[0]
Sigma = (1/m) * X_train_norm.T @ X_train_norm

print('Covariance matrix (5x5):')
print(Sigma.round(3))

In [ ]:
# STEP 3: Eigendecomposition
# eigenvalues  = how much variance each direction has
# eigenvectors = the directions (each column is one principal component)
eigenvalues, eigenvectors = np.linalg.eig(Sigma)

# STEP 4: Sort by eigenvalue descending (largest variance first)
order = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[order].real        # .real drops tiny imaginary rounding noise
eigenvectors = eigenvectors[:, order].real    # sort columns

print('Eigenvalues (sorted):')
for i, lam in enumerate(eigenvalues):
    pct = lam / eigenvalues.sum() * 100
    print('  PC%d: %.4f  (%.1f%% of variance)' % (i+1, lam, pct))

In [ ]:
# STEP 5: Project onto PC1
U1 = eigenvectors[:, :1]                      # shape (5, 1) — first principal direction
x1_train = (X_train_norm @ U1).flatten()       # shape (m,)  — 1D representation

explained_var = eigenvalues[0] / eigenvalues.sum()
print('PC1 explains %.1f%% of total variance' % (explained_var * 100))
print('x1_train shape:', x1_train.shape)
print('x1_train range: [%.3f, %.3f]' % (x1_train.min(), x1_train.max()))

---
## Problem 1(c) — Plot t vs x1, choose polynomial degree

Look at the scatter plot and decide: is the relationship linear, quadratic, cubic?

In [ ]:
# --- 1(c) ---
plt.figure(figsize=(7, 5))
plt.scatter(x1_train, t_train, color='steelblue', edgecolors='navy', s=40, alpha=0.8)
plt.xlabel('x1  (PC1 projection)')
plt.ylabel('t  (target)')
plt.title('t vs x1 — decide polynomial degree from this plot')
plt.grid(True, alpha=0.3)
plt.show()

# Observation: describe what you see — one bend suggests degree 2, two bends degree 3, etc.
print('Observation: the relationship appears quadratic (one bend).')
print('Selected degree: 2')

---
## Problem 1(d) — Polynomial regression + prediction uncertainty

### How the Normal Equation works

For degree `d`, build the design matrix Phi where each row is `[1, x, x^2, ..., x^d]`.
Then the optimal weights in one step (no iteration):

    W = (Phi^T @ Phi)^{-1} @ Phi^T @ t

This is the closed-form solution to minimising MSE. It comes from setting the
gradient of the cost to zero and solving for W.

### Uncertainty = RMSE

After fitting, compute residuals = t - t_predicted.
RMSE = sqrt(mean(residuals^2)) tells you the typical prediction error in the same units as t.

In [ ]:
# --- 1(d) ---
DEGREE = 2   # change this based on what you see in 1(c)

def build_design_matrix(x, degree):
    # Each row: [1, x, x^2, ..., x^degree]
    # Input:  x      shape (m,)
    # Output: Phi    shape (m, degree+1)
    return np.column_stack([x**i for i in range(degree + 1)])

Phi_train = build_design_matrix(x1_train, DEGREE)
print('Design matrix shape:', Phi_train.shape)
print('First row:', Phi_train[0].round(4), ' = [1, x, x^2]')

In [ ]:
# Normal equation: W = (Phi^T Phi)^{-1} Phi^T t
W = np.linalg.inv(Phi_train.T @ Phi_train) @ Phi_train.T @ t_train
print('Learned weights W:', W.round(4))

In [ ]:
# Predictions on training data
t_pred_train = Phi_train @ W

# Residuals and RMSE
residuals  = t_train - t_pred_train
sigma_pred = np.sqrt(np.mean(residuals**2))

print('RMSE (training):               %.4f' % sigma_pred)
print('Expected uncertainty (1 sigma): +/- %.4f' % sigma_pred)
print('95%% confidence interval (2 sigma): +/- %.4f' % (2*sigma_pred))

In [ ]:
# Plot: data + fit + uncertainty band
x_line    = np.linspace(x1_train.min() - 0.1, x1_train.max() + 0.1, 300)
Phi_line  = build_design_matrix(x_line, DEGREE)
t_line    = Phi_line @ W

plt.figure(figsize=(9, 5))
plt.fill_between(x_line, t_line - sigma_pred, t_line + sigma_pred,
                 alpha=0.2, color='red', label='+/- 1 sigma = +/- %.2f' % sigma_pred)
plt.fill_between(x_line, t_line - 2*sigma_pred, t_line + 2*sigma_pred,
                 alpha=0.1, color='red', label='+/- 2 sigma (95%%)')
plt.scatter(x1_train, t_train, color='steelblue', edgecolors='navy',
            s=40, alpha=0.8, zorder=5, label='Training data (t)')
plt.plot(x_line, t_line, 'r-', linewidth=2.5, zorder=4,
         label='Degree-%d fit (t_hat)' % DEGREE)
plt.xlabel('x1 (PC1)')
plt.ylabel('t')
plt.title('Polynomial regression (degree %d) — RMSE = %.3f' % (DEGREE, sigma_pred))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Problem 1(e) — Predict on test dataset

Apply the **exact same pipeline** to test data:
1. Normalise using **training** mu_X and sigma_X
2. Project using **training** eigenvectors U1
3. Build design matrix and multiply by **training** weights W

Never recompute statistics from the test set — that would put it on a different scale.

In [ ]:
# --- 1(e) ---

# Step 1: normalise test data with TRAINING mu_X and sigma_X
X_test_norm = (X_test - mu_X) / sigma_X

# Step 2: project onto PC1 with TRAINING eigenvectors
x1_test = (X_test_norm @ U1).flatten()

# Step 3: build design matrix and predict
Phi_test    = build_design_matrix(x1_test, DEGREE)
t_test_pred = Phi_test @ W

print('Predicted values for test set:')
print('%8s  %10s  %15s' % ('Sample', 'x1', 't_predicted'))
print('-' * 38)
for i, (xi, ti) in enumerate(zip(x1_test, t_test_pred)):
    print('%8d  %10.4f  %15.4f' % (i, xi, ti))

---
## Problem 2(d) — Sample the Laplace distribution

The PDF from the exam:

    p(x) = A * exp( -sqrt(2) * |x - mu| / lambda )

where A = 1 / (lambda * sqrt(2))  [derived in Part a]

### Why does the sampling recipe work? (Inversion sampling)

For the right half (x > mu=0), the CDF is F(x) = 1 - exp(-sqrt(2)*x/lambda).
Inverting: x = -(lambda/sqrt(2)) * log(1-u).
Since (1-u) is also Uniform[0,1], we write x = -(lambda/sqrt(2)) * log(u2).
Multiplying by s randomly assigns left or right of mu (50/50 via u1).

In [ ]:
# --- 2(d) ---
lam = 3     # lambda = 3 as specified
mu  = 0
N   = 50000

rng = np.random.default_rng(seed=42)

u1 = rng.uniform(0, 1, N)              # determines sign
u2 = rng.uniform(0, 1, N)              # determines distance from mu

# np.where(condition, value_if_true, value_if_false) — vectorised if/else
s = np.where(u1 < 0.5, 1, -1)

# Inversion formula: x = s * lambda * log(u2) / sqrt(2)
# log(u2) is negative since u2 in (0,1], so s flips the sign to both sides
x_samples = mu + s * lam * np.log(u2) / np.sqrt(2)

print('Generated %d samples' % N)
print('Sample mean:     %.4f  (expected: %d)' % (x_samples.mean(), mu))
print('Sample variance: %.4f  (expected lambda^2: %d)' % (x_samples.var(), lam**2))

In [ ]:
# Plot histogram vs theoretical PDF
x_plot = np.linspace(-15, 15, 500)
A   = 1 / (lam * np.sqrt(2))
pdf = A * np.exp(-np.sqrt(2) * np.abs(x_plot - mu) / lam)

plt.figure(figsize=(9, 5))
plt.hist(x_samples, bins=80, density=True,
         alpha=0.5, color='steelblue', label='Sampled histogram (N=%d)' % N)
plt.plot(x_plot, pdf, 'r-', linewidth=2.5, label='Theoretical PDF')
plt.xlabel('x')
plt.ylabel('Probability density')
plt.title('Laplace distribution: lambda=%d, mu=%d — histogram matches PDF' % (lam, mu))
plt.legend()
plt.xlim(-15, 15)
plt.grid(True, alpha=0.3)
plt.show()

---
## Problem 2(e) — Verify variance = lambda^2 for three different lambda

Generate a large sample for each lambda, compute np.var(x), compare to lambda^2.

In [ ]:
# --- 2(e) ---
rng = np.random.default_rng(seed=0)
mu  = 0
N   = 100000

print('%6s  %8s  %14s  %14s' % ('lambda', 'lambda^2', 'Measured var', 'Ratio var/lam^2'))
print('-' * 50)

for lam in [1, 3, 5]:
    u1  = rng.uniform(0, 1, N)
    u2  = rng.uniform(0, 1, N)
    s   = np.where(u1 < 0.5, 1, -1)
    x   = mu + s * lam * np.log(u2) / np.sqrt(2)

    measured_var = np.var(x)
    theoretical  = lam**2
    ratio        = measured_var / theoretical

    print('%6d  %8.1f  %14.4f  %14.4f' % (lam, theoretical, measured_var, ratio))

print()
print('All ratios close to 1.0 — variance equals lambda^2 for all tested values')

In [ ]:
# Visual confirmation: 3 distributions side by side
lambdas = [1, 3, 5]
colors  = ['#4caf50', '#2196f3', '#ff9800']
x_plot  = np.linspace(-20, 20, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
rng2 = np.random.default_rng(seed=7)

for ax, lam, color in zip(axes, lambdas, colors):
    u1 = rng2.uniform(0, 1, 30000)
    u2 = rng2.uniform(0, 1, 30000)
    s  = np.where(u1 < 0.5, 1, -1)
    xs = 0 + s * lam * np.log(u2) / np.sqrt(2)

    ax.hist(xs, bins=60, density=True, alpha=0.5, color=color)

    A_lam = 1 / (lam * np.sqrt(2))
    pdf   = A_lam * np.exp(-np.sqrt(2) * np.abs(x_plot) / lam)
    ax.plot(x_plot, pdf, 'k-', linewidth=2)

    v = np.var(xs)
    ax.set_title('lambda=%d  |  var=%.2f  |  lambda^2=%d' % (lam, v, lam**2))
    ax.set_xlabel('x')
    ax.set_xlim(-18, 18)
    ax.grid(True, alpha=0.2)

plt.suptitle('Laplace: measured variance is approx lambda^2', y=1.02)
plt.tight_layout()
plt.show()

---
## Paper answers for Problems 2(a–c)

### 2(a) — Find A

Require the integral over all x to equal 1. Split at x = mu using symmetry:

    integral = 2 * A * integral_0^inf exp(-sqrt(2)*u/lambda) du
             = 2 * A * [lambda/sqrt(2)]
             = A * lambda * sqrt(2) = 1

    => A = 1 / (lambda * sqrt(2))

### 2(b) — Mean

The PDF is symmetric around x = mu: p(mu + u) = p(mu - u) for all u.
For any symmetric distribution the mean equals the centre of symmetry.

    => E[X] = mu

### 2(c) — Variance = lambda^2

    Var(X) = 2A * integral_0^inf u^2 * exp(-alpha*u) du,  alpha = sqrt(2)/lambda

Using the standard result (integration by parts twice):

    integral_0^inf u^2 * e^{-alpha*u} du = 2 / alpha^3

    Var(X) = 2 * [1/(lambda*sqrt(2))] * [2 / (sqrt(2)/lambda)^3]
           = 2/(lambda*sqrt(2)) * 2*lambda^3 / (2*sqrt(2))
           = 2*lambda^3 / (lambda * 2)
           = lambda^2

---
## Exam time plan (120 minutes)

| Task | Time |
|------|------|
| Problem 2(a-c) on paper | 20 min |
| 1(a) Load + verify | 3 min |
| 1(b) PCA | 15 min |
| 1(c) Plot + choose degree | 5 min |
| 1(d) Regression + RMSE + plot | 15 min |
| 1(e) Test predictions | 5 min |
| 2(d) Sampling + histogram | 10 min |
| 2(e) Verify variance | 10 min |
| Upload + check labels | 5 min |
| Buffer | 32 min |

## Checklist before submitting

- Every cell labelled: `# --- 1(a) ---`, `# --- 1(b) ---` etc.
- PCA: normalise → covariance → eig → sort → project
- Always use training mu_X, sigma_X, U1 on test data
- Normal equation: W = inv(Phi.T @ Phi) @ Phi.T @ t
- RMSE = sqrt(mean((t - t_pred)^2))
- A = 1 / (lambda * sqrt(2)) for Laplace distribution
- Sampling: u1 for sign, u2 for distance, x = s * lam * log(u2) / sqrt(2)